In [1]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

train = pd.read_csv(root / "data" / "raw" / "train.csv")


In [3]:
numerical_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]
features = numerical_features + categorical_features

x = train[features]
y = train["Survived"]

x_train, x_valid, y_train, y_valid = train_test_split(
    x, 
    y, 
    test_size=0.20, 
    stratify=y,
    random_state=42,
)

In [4]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("importer", numerical_pipeline, numerical_features),
    ("encoder", categorical_pipeline, categorical_features)
])

modell = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

modell.fit(x_train, y_train)
predictions = modell.predict(x_valid)

majority_class = y_valid.mode()[0]
dummy_accuracy = (y_valid == majority_class).mean()
model_accuracy = accuracy_score(y_valid, predictions)

In [5]:
print(f"Dummy accuracy: {dummy_accuracy:.3f}")
print(f"Logistic Regression accuracy: {model_accuracy:.3f}")
print(f"Improvement: {model_accuracy - dummy_accuracy:+.3f}")

Dummy accuracy: 0.615
Logistic Regression accuracy: 0.804
Improvement: +0.190


## Result

Logistic Regression achieved **0.804** validation accuracy, improving 
by **+0.190** over the majority baseline. The comparison uses the 
same stratified split, making the result fair.